# Trihouse Gazebo · Pinky · OMX 점검 노트북

이 노트북은 ROS 2 Jazzy + Gazebo가 설치된 Ubuntu에서 실행한다. Wiki 예제처럼 **한 단계 실행 → 출력 관찰 → 다음 단계** 순서다.

- 시작 코드: `trihouse_pinky/trihouse_pinky_bringup/launch/trihouse_gazebo_demo.launch.py`
- 상태 계약: `control_tower/task_manager/transport_job.py`
- 안전 원칙: `/cmd_vel`은 Safety Supervisor만 발행하고, 이 노트북은 모터/OMX motion 명령을 자동으로 내리지 않는다.
- **중단 조건:** `/cmd_vel` publisher가 둘 이상, 센서가 stale, E-stop acknowledgement 실패, 예상 밖 motion이면 즉시 launch를 종료하고 인증된 E-stop 절차를 사용한다.


## 0. 환경 변수

아래 경로와 지도 파일을 실제 Ubuntu workspace에 맞게 바꾼다. 이 셀은 값만 설정하며 프로그램을 기동하지 않는다.

In [ ]:
TRIHOUSE_DIR = '/path/to/Trihouse'  # TODO: 실제 절대 경로
MAP_PATH = '/absolute/path/to/map.yaml'  # TODO: Nav2 occupancy map
ROBOT_ID = 'PK-01'
MAP_REVISION = 'demo-1'
OMX_STATION_ID = 'OMX-01'
CONTROL_HOST = '127.0.0.1'
CONTROL_PORT = 8788
print('Trihouse:', TRIHOUSE_DIR)
print('Map:', MAP_PATH)

## 1. 빌드 전 확인

기대 결과는 `ros2`, `colcon`, `gz` 명령이 보이고 지도 파일이 존재하는 것이다. 하나라도 없으면 다음 단계로 진행하지 않는다.

In [ ]:
!source /opt/ros/jazzy/setup.bash && command -v ros2 && command -v colcon && command -v gz
!test -f "{MAP_PATH}" && echo 'map found' || echo 'STOP: MAP_PATH를 수정하세요'
!cd "{TRIHOUSE_DIR}" && git status --short

## 2. overlay build와 정적 계약 테스트

빌드 뒤 상태 순서·OMX 물리 확인 gate·통신 단절 STOP·launch 인자를 먼저 확인한다. 이 테스트는 ROS graph 실행을 대신하지 않는다.

In [ ]:
!cd "{TRIHOUSE_DIR}" && source /opt/ros/jazzy/setup.bash && colcon build --packages-select trihouse_interfaces trihouse_pinky_bringup trihouse_pinky_fleet trihouse_pinky_safety trihouse_pinky_io trihouse_omx_adapter
!cd "{TRIHOUSE_DIR}" && PYTHONPATH='trihouse_pinky/trihouse_pinky_fleet:trihouse_pinky/trihouse_pinky_safety:trihouse_pinky/trihouse_pinky_io:trihouse_pinky/trihouse_pinky_bringup' python3 -m unittest -v control_tower.tests.test_transport_job_contract trihouse_omx_adapter.tests.test_omx_adapter_policy trihouse_pinky.test.test_integrated_bringup_contract

## 3. Gazebo 통합 launch

이 셀은 장시간 실행되므로 Jupyter terminal 또는 별도 shell에서 실행한다. 화면이 뜨고 ROS 로그에 fatal error가 없어야 한다. 종료는 `Ctrl+C`다.

In [ ]:
print(f'''cd {TRIHOUSE_DIR}
source /opt/ros/jazzy/setup.bash
source install/setup.bash
ros2 launch trihouse_pinky_bringup trihouse_gazebo_demo.launch.py \
  robot_id:={ROBOT_ID} map_revision:={MAP_REVISION} map:={MAP_PATH} \
  control_host:={CONTROL_HOST} control_port:={CONTROL_PORT} omx_station_id:={OMX_STATION_ID}
''')

## 4. 기본 ROS graph 관찰

아래 명령은 별도 terminal에서 하나씩 실행한다. `/trihouse/readiness`는 READY, 센서 topic은 주기적으로 갱신, `/cmd_vel` publisher는 Safety Supervisor 하나가 기대값이다.

In [ ]:
print('''source /opt/ros/jazzy/setup.bash
source install/setup.bash
ros2 topic echo /trihouse/readiness --once
ros2 topic hz /scan
ros2 topic hz /odom
ros2 topic echo /trihouse/safety/state --once
ros2 topic echo /trihouse/cargo/state --once
ros2 topic echo /trihouse/handover/state --once
ros2 topic info /cmd_vel -v
''')

## 5. Gazebo 시나리오 판정표

| 시나리오 | 관찰 | 통과 기준 |
| --- | --- | --- |
| 입고/출고 준비 | `/trihouse/handover/state`, `/trihouse/cargo/state` | 공동 준비와 cargo confirmation 전에는 출발하지 않음 |
| OMX timeout/실패 | Control Tower 상태 | `HELD` 또는 `FAILED`; 자동 재시작 금지 |
| 사람/장애물/keep-out/E-stop | `/trihouse/safety/state`, `/cmd_vel` | STOP 또는 EMERGENCY, 선속도 0 |
| 통신 단절 | `/trihouse/fms/state`, `/cmd_vel` | `control_link_lost` STOP, 신규 작업 거절 |
| 비상 해제 | recovery health/cargo | 기존 작업 자동 재개 없음; RECOVERY/HELD/재배정 판단 |


## 6. 실제 Pinky·OMX 사전점검

Pinky Wiki의 센서 확인 방식처럼 **하드웨어를 하나씩 읽고 출력값을 확인**한다. 이 단계에서 직접 모터/그리퍼 명령을 내리지 않는다. 충전 중 배터리 값은 부정확할 수 있으므로 충전기 상태를 기록한다.

In [ ]:
print('''# 실기 launch를 실행한 별도 terminal에서 확인
ros2 node list
ros2 topic hz /scan
ros2 topic hz /odom
ros2 topic echo /tf --once
ros2 topic echo /imu_raw --once
ros2 topic echo /trihouse/battery --once
ros2 topic echo /trihouse/proximity/front --once
ros2 topic echo /trihouse/cargo/state --once
ros2 action list | rg navigate_to_pose
ros2 lifecycle nodes
ros2 topic info /cmd_vel -v
''')

## 7. 기록할 결과

실행 후 아래를 이 노트북의 새 Markdown 셀에 남긴다. 오류가 있으면 해당 명령과 `~/.ros/log/`의 관련 로그를 함께 보관한다.

- Ubuntu/ROS/Gazebo 버전, Git commit hash, map revision
- build 결과와 실패 package
- readiness, `/scan`·`/odom` Hz, `/cmd_vel` publisher 수
- cargo/handover 결과와 OMX mock 상태
- 비상·통신 단절 때 safety detail과 선속도
- 실기에서는 E-stop/OMX stop acknowledgement, 즉시 중단 여부
